# Bonus — Pretraining GPT on a 1 GB Project Gutenberg corpus

This notebook adapts the book's [Project Gutenberg pretraining bonus](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/03_bonus_pretraining_on_gutenberg) to this repository's typed, nbdev-exported components.

## Experiment plan

The upstream directory contains a workflow, not a bundled dataset. Its original downloader fetches roughly 50 GB before preprocessing. This adaptation streams a public English Gutenberg mirror and stops after preparing **1,000,000,000 UTF-8 bytes**.

The upstream structure is preserved: prepare manageable text shards, load one shard at a time, keep one model and optimizer across files, evaluate and generate periodically, save resumable checkpoints, report ETA, and make one epoch over the prepared corpus.

### A mental model of the hierarchy

```text
1 GB corpus
  -> approximately 20 shards stored on disk
      -> many fixed-length token windows in each shard
          -> several windows grouped into each batch
              -> one optimizer step per training batch
```

A **shard** is a storage and memory-management unit. A **batch** is the group used for one gradient update. An **epoch** is complete only after every training batch from every shard has been processed once.

The model is created once. Its parameters and AdamW state continue changing across shard boundaries; loading a shard does not start a new training run.

The notebook is built but unexecuted. Runtime depends on token count, GPU load, and the from-scratch attention implementation, so one night is a target rather than a guarantee. Gutenberg texts are generally public domain in the United States but may have different status elsewhere.

In [11]:
import time
from pathlib import Path

import tiktoken
import torch
from datasets import IterableDataset, load_dataset
from torch.utils.data import DataLoader

from build_llms_from_scratch_companion.data import create_dataloader_v1
from build_llms_from_scratch_companion.model import GPTConfig, GPTModel
from build_llms_from_scratch_companion.training import (
    calc_loss_batch,
    evaluate_model,
    generate_and_print_sample,
    plot_losses,
)

## Paths and dataset budget

The total prepared-text budget remains one decimal gigabyte:

```text
training text     up to 990 MB
validation text   up to  10 MB
combined budget        1 GB
```

Approximately 50 MB training shards produce about 20 files. The fixed validation file contains complete books that never enter training. Runtime artifacts live under Git-ignored `data/` and `checkpoints/` directories. A success marker distinguishes complete preparation from interrupted output.

In [ ]:
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()
)
data_dir = repo_root / "data" / "gutenberg_1gb"
checkpoint_dir = repo_root / "checkpoints" / "gutenberg_1gb"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

DATASET_ID = "AdhyanshVerma/pg-en"
DATASET_REVISION = "fb11819399b61842257ee3dc64b89547f6aa358d"
TOTAL_DATA_BUDGET_BYTES = 1_000_000_000
VALIDATION_BUDGET_BYTES = 10_000_000
TRAIN_DATA_BUDGET_BYTES = TOTAL_DATA_BUDGET_BYTES - VALIDATION_BUDGET_BYTES
SHARD_TARGET_BYTES = 50_000_000
DOCUMENT_SEPARATOR = "\n<|endoftext|>\n"

## Streaming and preparing 1 GB

Hugging Face streaming retrieves books lazily. Preparation first reserves up to 10 MB of **complete books** for a fixed validation file, then writes up to 990 MB of different books into training shards. GPT-2's `<|endoftext|>` token separates documents.

This book-level separation prevents exact passages—and their surrounding book—from appearing in both splits. Ten megabytes is only 1% of the corpus but still represents millions of token positions, far more than periodic evaluation needs.

Byte accounting includes UTF-8 text and separators. The total may fall slightly below 1 GB because a complete book is never cut merely to fill the remaining budget.

In [ ]:
def prepare_gutenberg_shards(
    dataset: IterableDataset,
    output_dir: Path,
    train_budget_bytes: int,
    validation_budget_bytes: int,
    shard_target_bytes: int,
    separator: str,
    signature: str,
) -> tuple[list[Path], Path]:
    """Prepare training shards and a fixed whole-book validation file.

    Args:
        dataset: Streaming dataset whose rows contain a `text` field.
        output_dir: Directory where prepared plaintext files are written.
        train_budget_bytes: Maximum UTF-8 bytes in all training shards.
        validation_budget_bytes: Maximum UTF-8 bytes in validation text.
        shard_target_bytes: Approximate maximum bytes in one training shard.
        separator: Text inserted between complete books.
        signature: Preparation settings stored in the success marker.

    Returns:
        Sorted training-shard paths and the fixed validation-file path.

    Raises:
        ValueError: If training or validation receives no nonempty text.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    marker = output_dir / "_SUCCESS"
    validation_path = output_dir / "validation.txt"
    existing = sorted(output_dir.glob("combined_*.txt"))
    if (
        marker.exists()
        and marker.read_text(encoding="utf-8") == signature
        and validation_path.exists()
        and existing
    ):
        return existing, validation_path

    for path in existing:
        path.unlink()
    validation_path.unlink(missing_ok=True)
    marker.unlink(missing_ok=True)

    separator_bytes = len(separator.encode("utf-8"))
    validation_books: list[str] = []
    validation_bytes = 0
    current_books: list[str] = []
    current_bytes = train_bytes = shard_index = 0
    shard_paths: list[Path] = []

    def flush_shard() -> None:
        """Write the current training-book buffer and reset it.

        Returns:
            None.
        """
        nonlocal current_books, current_bytes, shard_index
        if not current_books:
            return
        shard_index += 1
        path = output_dir / f"combined_{shard_index:04d}.txt"
        path.write_text(separator.join(current_books), encoding="utf-8")
        shard_paths.append(path)
        current_books, current_bytes = [], 0

    for row in dataset:
        text = row.get("text")
        if not isinstance(text, str) or not text.strip():
            continue
        text_bytes = len(text.encode("utf-8"))

        # Fill validation first with complete books, then keep it fixed.
        if validation_bytes < validation_budget_bytes:
            boundary = separator_bytes if validation_books else 0
            additional = boundary + text_bytes
            if validation_bytes + additional <= validation_budget_bytes:
                validation_books.append(text)
                validation_bytes += additional
                continue

        boundary = separator_bytes if current_books else 0
        additional = boundary + text_bytes
        if train_bytes + additional > train_budget_bytes:
            break
        if current_books and current_bytes + additional > shard_target_bytes:
            flush_shard()
            additional = text_bytes
        current_books.append(text)
        current_bytes += additional
        train_bytes += additional

    flush_shard()
    if not validation_books or not shard_paths:
        raise ValueError("Training and validation text must both be prepared")
    validation_path.write_text(
        separator.join(validation_books),
        encoding="utf-8",
    )
    marker.write_text(signature, encoding="utf-8")
    return shard_paths, validation_path

### Preparing or reusing shards

`load_dataset(..., streaming=True)` retrieves records lazily. A matching `_SUCCESS` marker makes reruns reuse the local shards immediately.

Finding some `combined_*.txt` files does not prove preparation finished; a network failure could leave a plausible-looking incomplete directory. The marker signature records the dataset revision and byte settings, preventing stale shards from being silently reused.

Preparation and pretraining are separate phases. If training crashes, text preparation does not need to run again.

In [ ]:
signature = (
    f"{DATASET_ID}|{DATASET_REVISION}|{TRAIN_DATA_BUDGET_BYTES}|"
    f"{VALIDATION_BUDGET_BYTES}|{SHARD_TARGET_BYTES}"
)
gutenberg_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
    revision=DATASET_REVISION,
)
shard_paths, validation_path = prepare_gutenberg_shards(
    gutenberg_stream,
    data_dir,
    TRAIN_DATA_BUDGET_BYTES,
    VALIDATION_BUDGET_BYTES,
    SHARD_TARGET_BYTES,
    DOCUMENT_SEPARATOR,
    signature,
)
train_bytes = sum(path.stat().st_size for path in shard_paths)
validation_bytes = validation_path.stat().st_size
print("Training shards:", len(shard_paths))
print("Training text (GB):", f"{train_bytes / 1_000_000_000:.3f}")
print(
    "Validation text (MB):",
    f"{validation_bytes / 1_000_000:.1f}",
)
print(
    "Total prepared text (GB):",
    f"{(train_bytes + validation_bytes) / 1_000_000_000:.3f}",
)

## GPT-2 small on a 12 GB GPU

Keep GPT-2 small's 768-dimensional embeddings, 12 heads, and 12 blocks. Reduce only `context_length` from 1,024 to 256 so the from-scratch attention implementation and AdamW state fit more comfortably on the RTX 4070 SUPER.

```text
input_batch   (batch_size=8, num_tokens=256)
target_batch  (batch_size=8, num_tokens=256)
logits        (batch_size=8, num_tokens=256, vocab_size=50_257)
```

### Why training uses much more memory than inference

GPU memory holds model parameters, gradients, AdamW moving averages, backward activations, attention matrices, and vocabulary logits. AdamW stores two running statistics for most parameters. Attention memory grows quickly with `num_tokens` because each head compares token positions pairwise.

If CUDA runs out of memory, lower `BATCH_SIZE` first. This reduces examples processed simultaneously without changing the model architecture. A smaller batch causes more optimizer steps and may increase runtime.

In [ ]:
GUTENBERG_CONFIG = GPTConfig(
    vocab_size=50257,
    context_length=256,
    emb_dim=768,
    num_heads=12,
    num_layers=12,
    dropout_rate=0.1,
    qkv_bias=False,
)
BATCH_SIZE = 8
NUM_EPOCHS = 1
LEARNING_RATE = 5e-4
EVAL_FREQ = 100
EVAL_ITER = 5
PRINT_SAMPLE_FREQ = 1_000
SAVE_CHECKPOINT_FREQ = 5_000
START_CONTEXT = "Every effort moves you"
RESUME_FROM_CHECKPOINT = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Loading one training shard at a time

Reading one approximately 50 MB training shard bounds host memory. `GPTDatasetV1` tokenizes it and materializes shifted input and target windows; those objects can be released before the next shard is loaded.

The validation loader is created once from `validation.txt`, uses books that never enter training, remains unshuffled, and is reused throughout the entire run. This makes validation losses comparable across shard boundaries: changes mainly reflect the model rather than a changing evaluation sample.

`EVAL_ITER=5` averages five fixed validation batches at each evaluation. This is still a sample of the validation set for speed, but it is more stable than one batch and does not require reserving 100 MB of text.

In [ ]:
def create_training_loader(
    shard_path: Path,
    batch_size: int,
    context_length: int,
) -> DataLoader[tuple[torch.Tensor, torch.Tensor]]:
    """Create one shuffled training loader from one plaintext shard.

    Args:
        shard_path: Path of one prepared Gutenberg training shard.
        batch_size: Independent token windows in a complete batch.
        context_length: Token IDs in every input and target window.

    Returns:
        Training loader yielding token-ID tensors shaped
        `(batch_size, num_tokens)` for complete batches.
    """
    text_data = shard_path.read_text(encoding="utf-8")
    return create_dataloader_v1(
        text_data,
        batch_size=batch_size,
        max_length=context_length,
        stride=context_length,
        shuffle=True,
        drop_last=True,
        num_workers=0,
    )


validation_text = validation_path.read_text(encoding="utf-8")
val_loader = create_dataloader_v1(
    validation_text,
    batch_size=BATCH_SIZE,
    max_length=GUTENBERG_CONFIG.context_length,
    stride=GUTENBERG_CONFIG.context_length,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)
print("Fixed validation batches:", len(val_loader))

## Saving resumable state

A checkpoint is a snapshot of training state, not only learned weights. It stores model parameters, AdamW moving averages, the resume shard, global step, cumulative token count, and aligned loss histories. Restoring only model weights while recreating AdamW would discard the optimizer's memory of previous gradients.

### Exact versus approximate resumption

A completed-shard checkpoint resumes from the next file. A mid-shard checkpoint resumes from that shard's beginning because the current `DataLoader` iterator and shuffled order are not serialized. Some text may therefore be seen twice after interruption.

Perfect mid-batch restoration would also save sampler position and Python, PyTorch CPU, and CUDA random states. Dedicated training frameworks commonly handle that complexity. Shard-level recovery is a practical educational compromise.

GPT-2 small plus AdamW state can occupy several gigabytes. Check available disk space before leaving the run unattended.

In [ ]:
def save_training_checkpoint(
    checkpoint_path: Path,
    model: GPTModel,
    optimizer: torch.optim.Optimizer,
    next_shard_index: int,
    global_step: int,
    tokens_seen: int,
    train_losses: list[float],
    val_losses: list[float],
    track_tokens_seen: list[int],
) -> None:
    """Save model, optimizer, progress, and metric history.

    Args:
        checkpoint_path: Destination of the serialized checkpoint.
        model: GPT model whose parameters are saved.
        optimizer: Optimizer whose adaptive state is saved.
        next_shard_index: Shard index used when resuming.
        global_step: Number of the most recent optimizer step.
        tokens_seen: Cumulative input token positions processed.
        train_losses: Recorded training losses.
        val_losses: Recorded validation losses.
        track_tokens_seen: Token counts aligned with loss histories.

    Returns:
        None.
    """
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "next_shard_index": next_shard_index,
            "global_step": global_step,
            "tokens_seen": tokens_seen,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "track_tokens_seen": track_tokens_seen,
        },
        checkpoint_path,
    )

## Training across all shards for one epoch

One model and optimizer continue across every training file. Each batch clears gradients, computes next-token loss, backpropagates, and applies AdamW. `tokens_seen` and `global_step` continue across shard boundaries.

Periodic evaluation compares a fresh sample from the current training shard with the same five batches from the fixed whole-book validation loader. `model.eval()` disables dropout and `torch.no_grad()` disables gradient recording; training mode is restored afterward.

Because validation books never change, the validation curve is easier to interpret over the full epoch. ETA remains approximate because tokenization, evaluation, checkpoints, and GPU contention vary by shard.

In [ ]:
def train_on_gutenberg_shards(
    model: GPTModel,
    optimizer: torch.optim.Optimizer,
    shard_paths: list[Path],
    tokenizer: tiktoken.Encoding,
    val_loader: DataLoader[tuple[torch.Tensor, torch.Tensor]],
    device: torch.device,
    checkpoint_path: Path,
    start_shard_index: int,
    global_step: int,
    tokens_seen: int,
    train_losses: list[float],
    val_losses: list[float],
    track_tokens_seen: list[int],
) -> tuple[list[float], list[float], list[int]]:
    """Train for one epoch across prepared Gutenberg shards.

    Args:
        model: GPT model updated throughout the corpus.
        optimizer: AdamW optimizer shared across every shard.
        shard_paths: Ordered plaintext shards comprising the corpus.
        tokenizer: GPT-2 tokenizer used for generated samples.
        val_loader: Fixed loader of held-out whole-book validation text.
        device: CPU or CUDA device shared by model and batches.
        checkpoint_path: Path overwritten with resumable state.
        start_shard_index: Zero-based shard index where training resumes.
        global_step: Number of the most recent optimizer step.
        tokens_seen: Cumulative input token positions already processed.
        train_losses: Existing training-loss history when resuming.
        val_losses: Existing validation-loss history when resuming.
        track_tokens_seen: Token counts aligned with loss history.

    Returns:
        Updated training losses, validation losses, and token counts.
    """
    run_start = time.perf_counter()
    shard_times: list[float] = []
    shard_index = start_shard_index

    try:
        for shard_index in range(start_shard_index, len(shard_paths)):
            shard_start = time.perf_counter()
            shard_path = shard_paths[shard_index]
            print(f"Shard {shard_index + 1}/{len(shard_paths)}: {shard_path.name}")
            train_loader = create_training_loader(
                shard_path,
                BATCH_SIZE,
                GUTENBERG_CONFIG.context_length,
            )
            model.train()

            for input_batch, target_batch in train_loader:
                # input_batch and target_batch: (batch_size, num_tokens)
                optimizer.zero_grad()
                # loss: scalar tensor with shape ()
                loss = calc_loss_batch(input_batch, target_batch, model, device)
                loss.backward()
                optimizer.step()
                tokens_seen += input_batch.numel()
                global_step += 1

                if global_step % EVAL_FREQ == 0:
                    train_loss, val_loss = evaluate_model(
                        model, train_loader, val_loader, device, EVAL_ITER
                    )
                    train_losses.append(train_loss)
                    val_losses.append(val_loss)
                    track_tokens_seen.append(tokens_seen)
                    print(f"Step {global_step:07d}: train {train_loss:.3f}, val {val_loss:.3f}")

                if global_step % PRINT_SAMPLE_FREQ == 0:
                    generate_and_print_sample(model, tokenizer, device, START_CONTEXT)

                if global_step > 0 and global_step % SAVE_CHECKPOINT_FREQ == 0:
                    # A mid-shard resume repeats this shuffled shard.
                    save_training_checkpoint(
                        checkpoint_path,
                        model,
                        optimizer,
                        shard_index,
                        global_step,
                        tokens_seen,
                        train_losses,
                        val_losses,
                        track_tokens_seen,
                    )

            # A completed-shard resume starts at the next shard.
            save_training_checkpoint(
                checkpoint_path,
                model,
                optimizer,
                shard_index + 1,
                global_step,
                tokens_seen,
                train_losses,
                val_losses,
                track_tokens_seen,
            )
            shard_seconds = time.perf_counter() - shard_start
            shard_times.append(shard_seconds)
            remaining = len(shard_paths) - shard_index - 1
            eta_seconds = sum(shard_times) / len(shard_times) * remaining
            print(f"Shard time: {shard_seconds / 3600:.2f} h | ETA: {eta_seconds / 3600:.2f} h")

    except KeyboardInterrupt:
        save_training_checkpoint(
            checkpoint_path,
            model,
            optimizer,
            shard_index,
            global_step,
            tokens_seen,
            train_losses,
            val_losses,
            track_tokens_seen,
        )
        print(f"Interrupted; saved {checkpoint_path}")

    elapsed = time.perf_counter() - run_start
    print(f"Elapsed time: {elapsed / 3600:.2f} h")
    return train_losses, val_losses, track_tokens_seen

## Initializing or resuming

Initialize GPT-2 small and AdamW on CUDA. If `RESUME_FROM_CHECKPOINT=True` and `latest.pt` exists, restore model and optimizer state together with counters and histories. Otherwise, begin from reproducible random weights and the first shard.

Rerunning this notebook is therefore intentional: cached shards are reused and the latest training state is loaded. To deliberately start over, remove or rename `checkpoints/gutenberg_1gb/latest.pt` before initialization.

A checkpoint must be loaded into the same architecture that created it. Changing model dimensions makes its stored tensors incompatible.

In [ ]:
latest_checkpoint_path = checkpoint_dir / "latest.pt"
torch.manual_seed(123)
model = GPTModel(GUTENBERG_CONFIG).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.1,
)
tokenizer = tiktoken.get_encoding("gpt2")
start_shard_index, global_step, tokens_seen = 0, -1, 0
train_losses: list[float] = []
val_losses: list[float] = []
track_tokens_seen: list[int] = []

if RESUME_FROM_CHECKPOINT and latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_shard_index = checkpoint["next_shard_index"]
    global_step = checkpoint["global_step"]
    tokens_seen = checkpoint["tokens_seen"]
    train_losses = checkpoint["train_losses"]
    val_losses = checkpoint["val_losses"]
    track_tokens_seen = checkpoint["track_tokens_seen"]
    print("Resuming at shard:", start_shard_index + 1)
else:
    print("Starting a new one-epoch run")

## Starting pretraining

This is the expensive cell. It processes every prepared shard once unless resuming. On interruption, it saves progress before returning.

### Overnight preflight checklist

Before leaving it unattended, verify that:

- preparation reported close to 1 GB and the expected shard count;
- `device` is `cuda`, not `cpu`;
- enough disk remains for data and multi-gigabyte checkpoints;
- Windows sleep or automatic restart will not suspend the process;
- no other application occupies most GPU memory;
- the first evaluation produces finite losses rather than `nan` or `inf`; and
- an early checkpoint is written successfully.

Loss need not decrease at every evaluation because batches differ. Persistent `nan`, exploding loss, CUDA out-of-memory errors, or continually worsening validation loss are reasons to stop and investigate.

Do not run this cell until preparation is complete and the reported device is CUDA.

In [ ]:
if device.type != "cuda":
    raise RuntimeError("CUDA is required for the planned overnight run")

train_losses, val_losses, track_tokens_seen = train_on_gutenberg_shards(
    model,
    optimizer,
    shard_paths,
    tokenizer,
    val_loader,
    device,
    latest_checkpoint_path,
    start_shard_index,
    global_step,
    tokens_seen,
    train_losses,
    val_losses,
    track_tokens_seen,
)

## Saving final weights and plotting loss

Save model-only weights separately from the resumable checkpoint. The model-only file is smaller and sufficient for inference; `latest.pt` is needed to continue AdamW training.

Plot losses against progress through the epoch and cumulative token positions. Falling training and validation losses indicate useful learning. Falling training loss with rising validation loss suggests overfitting or a distribution difference. Generated text is a qualitative check, but fluent-looking output alone does not prove broad knowledge or generalization.

The plotting guard handles interruption before the first evaluation.

In [ ]:
final_model_path = checkpoint_dir / "gutenberg_1gb_final.pt"
torch.save(model.state_dict(), final_model_path)
print("Saved:", final_model_path)

if train_losses:
    epochs_seen = torch.linspace(0, NUM_EPOCHS, len(train_losses))
    plot_losses(epochs_seen, track_tokens_seen, train_losses, val_losses)
if device.type == "cuda":
    print(
        "Peak CUDA memory (GB):",
        f"{torch.cuda.max_memory_allocated() / 1e9:.2f}",
    )

## Summary

This notebook prepares a bounded 1 GB Gutenberg corpus as approximately 990 MB of training shards plus a fixed 10 MB whole-book validation set. It trains across every shard once while retaining model and optimizer state, evaluates against stable held-out books, generates samples, estimates ETA, and writes resumable checkpoints.

The design highlights that shards, batches, optimizer steps, and epochs are different units; validation must remain independent and stable; and long-running training requires planning for memory, disk, monitoring, interruption, and recovery.

This remains an educational single-GPU implementation without Flash Attention, mixed precision, gradient accumulation, distributed training, or pretokenized shards.